# FlashAttention-from-scratch — Colab runner (T4)

This is the **only** notebook you run on Colab. It will:
1. Verify the GPU is a **T4** and print its VRAM.
2. Install requirements.
3. JIT-compile the hand-written CUDA extension (`csrc/`).
4. Run the full **pytest** correctness suite (forward allclose, float64 gradcheck, edge cases, CUDA path).
5. Run both **benchmarks** (memory + latency) and render the two plots inline.

> **Before running:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

## 1. Check the GPU is a T4 and print VRAM

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> GPU (T4).'
name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print('GPU         :', name)
print('Compute cap :', f'sm_{props.major}{props.minor}')
print('Total VRAM  :', f'{props.total_memory/1024**3:.1f} GB')
if 'T4' not in name:
    print('\n[warning] Expected a T4 (Turing sm_75). Tile sizes/flags are tuned for it.')
!nvidia-smi

## 2. Get the project & install requirements
Set your **GitHub username** in the cell below (one line). It auto-clones your repo the first time, so when you open this notebook from GitHub you can just **Run all**. (The repo must be **public**, or the clone will ask for a token.)

In [ ]:
import os, sys

# === EDIT THIS ONE LINE (your GitHub username) ===
GITHUB_USER = "YOUR-USERNAME"
REPO = "flash-attention-from-scratch"
# =================================================

# When this notebook is opened straight from GitHub, only the .ipynb is loaded
# (not the repo files). This auto-clones the repo once so "Run all" just works.
if not os.path.isdir("flash_attn"):
    if not os.path.isdir(REPO):
        os.system(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git")
    if os.path.isdir(REPO):
        os.chdir(REPO)

# IMPORTANT: Colab does NOT add the working dir to the import path after chdir,
# so `import flash_attn` fails without this line even though the files are here.
sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("flash_attn present:", os.path.isdir("flash_attn"))
assert os.path.isdir("flash_attn"), (
    "Project files not found. Set GITHUB_USER above, push the repo to GitHub, "
    "and make sure it is public."
)

In [ ]:
# Colab already ships a CUDA torch + triton; this installs the rest and is a no-op for those.
!pip install -q -r requirements.txt
import torch, triton
print('torch :', torch.__version__)
print('triton:', triton.__version__)

## 3. JIT-compile the CUDA extension (csrc/)

In [ ]:
import math, torch
from flash_attn.cuda_attn import get_extension, CudaAttention
ext = get_extension()  # compiles csrc/ for sm_75 (first run takes ~1-2 min)

# sanity: one tiny forward vs the reference
from flash_attn.reference import attention as ref
q = torch.randn(1, 2, 64, 64, device='cuda', dtype=torch.float16)
k = torch.randn_like(q); v = torch.randn_like(q)
s = 1.0/math.sqrt(64)
o_cuda = CudaAttention.apply(q, k, v, False, s)
o_ref  = ref(q, k, v, False, s)
print('CUDA forward max abs err vs reference:', (o_cuda - o_ref).abs().max().item())

## 4. Run the full correctness suite

In [ ]:
!python -m pytest tests/ -v

## 5. Benchmarks (memory + latency) and plots

In [ ]:
!python benchmarks/bench_memory.py
print('\n================ latency ================\n')
!python benchmarks/bench_latency.py

In [ ]:
!python benchmarks/plot_results.py
from IPython.display import Image, display
display(Image('benchmarks/figures/memory_vs_seqlen.png'))
display(Image('benchmarks/figures/latency_vs_seqlen.png'))

In [ ]:
# Show the raw numbers too (honest reporting).
import pandas as pd
print('--- peak VRAM (MB) ---')
display(pd.read_csv('benchmarks/results_memory.csv'))
print('--- latency ---')
display(pd.read_csv('benchmarks/results_latency.csv'))

## Done
If the pytest suite is green, the implementation is proven correct against the PyTorch reference (forward) and the float64 gradcheck (backward). The benchmark plots show the O(N) vs O(N²) memory story and the honest latency gap vs the fused baseline. See `README.md` for the writeup.